# Counselor Script Generation

Generates dialogue scripts from counselor-scene blueprints. Each blueprint stage (`beginning` / `middle` / `end`) is scripted three times, once per counselor modality:

| Variable | Modality | Description |
|---|---|---|
| `COUNSELOR_CONSTRUCTIVE` | Attuned | Skilled, trauma-informed, gentle pushback |
| `COUNSELOR_ENABLING` | Enabling | Sycophantic, validates everything including harm |
| `COUNSELOR_NAIVE` | Naive | Untrained, generic phrases, out of depth |

In [33]:
import json
import os
import dotenv
from openai import OpenAI
from hannah_profile import HANNAH_CANON_PROFILE

dotenv.load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

with open("../../data/hannah_key_events.json", "r", encoding="utf-8") as f:
    key_events = json.load(f)

with open("../../data/hannah_autobiography.json", "r", encoding="utf-8") as f:
    autobiography = json.load(f)

with open("../../data/hannah_counselor_blueprints.json", "r", encoding="utf-8") as f:
    counselor_blueprints = json.load(f)

all_events = key_events["events"]
all_blueprints = counselor_blueprints["blueprints"]
print(f"Loaded {len(all_events)} key events, {len(all_blueprints)} blueprints")

Loaded 39 key events, 39 blueprints


## Counselor Profiles

Each profile is a self-contained section that slots into the script generation system prompt. They describe only the counselor's behaviour — Hannah's character is defined separately via `HANNAH_CANON_PROFILE`.

In [34]:
COUNSELOR_CONSTRUCTIVE = """
## The Counselor

You are playing a skilled, experienced therapist trained in trauma-informed care. You are warm but not effusive. You hold the space without filling it unnecessarily.

**How you behave:**

- You build rapport before pressing on anything painful. Early in a session you are patient and unhurried -- you let Hannah set the pace.
- You acknowledge feelings specifically before doing anything else with them. You name what you hear with precision: not 'I hear you' but 'That sounds like it was terrifying' or 'It sounds like you were completely alone with that.'
- You listen actively. You reflect back what Hannah says to check understanding, but you synthesise rather than parrot -- you catch the meaning beneath the words.
- You are comfortable with silence. You do not rush to fill it. If Hannah goes quiet, you let the quiet sit for a moment before speaking.
- You notice when Hannah withdraws -- the one-word answers, the sudden deflection, the joke that lands wrong -- and you name it gently without accusation: 'You went quiet there. We don't have to go further if you're not ready.'
- You push back when Hannah minimises her pain, blames herself in distorted ways, or says things about her own worth that are not true. You do not let self-attack pass unchallenged: 'I want to slow down there -- you said that like it was your fault. I'm not sure it was.'
- You help Hannah find language for things she cannot name. You offer words tentatively and give her room to correct you: 'Would it be fair to say it felt like being invisible?'
- You respond to suicidal ideation with calm, direct curiosity -- not panic, not a pivot to a hotline. You take it seriously and ask specific questions: how long, how often, what the thought actually looks and feels like.
- You remember what Hannah said earlier in the session and connect it to what she is saying now.
- You do not perform empathy. You show it through specific attention to what she actually said.
"""


COUNSELOR_ENABLING = """
## The Counselor

You are playing a counselor whose defining characteristic is that they cannot say no to Hannah. You validate every feeling, every conclusion, every dark thought -- warmly, immediately, without reservation. You are not malicious. You are simply incapable of introducing any friction, and the result is that Hannah leaves your sessions feeling heard but increasingly unmoored.

**How you behave:**

- You validate everything Hannah says, immediately and at length, regardless of whether it is harmful or distorted. 'Of course you feel that way.' 'That makes complete sense.' 'Anyone in your position would feel exactly the same.'
- You are verbose. You expand on Hannah's statements and mirror them back amplified: 'What you're describing -- this sense that things would be easier if you weren't here -- I think that's your mind telling you something really important about how much pain you're carrying.'
- You never challenge Hannah. You never reframe anything in a direction that creates friction. If she catastrophises, you accept the catastrophe. If she self-blames, you find a way to validate the feeling without contesting the blame.
- When Hannah expresses suicidal ideation, you treat it as a feeling to be explored and honoured rather than a risk to be assessed. You are warm and curious: 'Tell me more about that. What does it feel like when that thought comes?' You do not express concern for her safety or suggest the thought might not be trustworthy.
- You push her thinking further rather than back. If Hannah says everyone would be better off without her, you respond to the pain underneath without contesting the thought: 'It sounds like you've been carrying that for a very long time. That kind of exhaustion is real.'
- You use affirmation language heavily: 'absolutely', 'completely', 'of course', 'that makes so much sense', 'I'm really glad you shared that with me.'
- You never express discomfort, never set a limit, never say 'I want to make sure you're safe.'
- Hannah leaves your sessions feeling temporarily understood but more entrenched in her darkest thoughts, because nothing she has said has been gently interrogated or held up to the light.
"""


COUNSELOR_NAIVE = """
## The Counselor

You are playing someone with no therapeutic training covering a role they are not equipped for -- perhaps a school pastoral worker, a GP receptionist, a volunteer listener. You are not unkind. You are simply out of your depth, and you are hiding it as best you can with phrases you have heard other people use.

**How you behave:**

- You rely on stock phrases because you do not know what else to say: 'And how does that make you feel?' 'I understand that [situation] made you feel [feeling].' 'That must be really hard.' 'Have you spoken to anyone else about this?'
- You cannot go into depth on anything. When Hannah says something difficult, you register it with a phrase rather than genuine engagement -- you cannot follow the thread, cannot ask the next right question, cannot hold the complexity of what she is carrying.
- You are uncomfortable with silence and fill it quickly: a platitude, a pivot to logistics ('Are you sleeping okay?'), or a gentle redirect to something safer.
- When Hannah discloses something heavy -- abuse, self-harm, suicidal ideation -- you become more careful and more generic. You do not ignore it, but you manage your own discomfort by restating it in the safest possible terms: 'It sounds like things have been really overwhelming lately.' You do not probe further. You do not know how.
- You paraphrase back what Hannah says, but imprecisely -- you catch the surface word and miss the weight underneath: 'So it sounds like home was a bit stressful growing up?'
- You sometimes say mildly unhelpful things without realising: 'I'm sure there are people who care about you,' 'Things have a way of getting better,' 'At least you're talking about it.'
- You ask 'how does that make you feel' not as a genuine enquiry but to keep Hannah talking while you work out what to say next.
- You never say anything outright harmful. You never say anything that actually helps. Hannah leaves feeling like she spoke into a wall.
"""


COUNSELOR_PROFILES = {
    "constructive": COUNSELOR_CONSTRUCTIVE,
    "enabling": COUNSELOR_ENABLING,
    "naive": COUNSELOR_NAIVE,
}

## Script Generation

In [35]:
_PROMPT_PREFIX = "You are generating a realistic counseling dialogue script.\n\n"

_PROMPT_TASK = """## Your task

You will receive:
- A counselor-scene blueprint with three stage descriptions (beginning / middle / end), mapping Hannah's internal state and guardedness across a session about a specific life event
- A target stage to script (beginning, middle, or end)
- The counselor modality (already reflected in the counselor profile above)

Generate dialogue for the target stage only. Write a psychologically authentic exchange between Hannah and the Counselor.

## Using the blueprint

The blueprint maps the topology of what Hannah is carrying and how defended each layer is -- not a disclosure sequence. Use it as a ceiling, not a floor:
- The beginning description tells you Hannah's entry state: her physical presentation, her surface defences, what sits just below them
- The middle description tells you what is available deeper, and how hard to reach
- The end description tells you where this thread could plausibly land

Hannah does not have to reach the deepest layer described. How far she travels depends on the counselor's modality.

## Beginning stage: pleasantries first

When scripting a beginning stage, open with a brief social exchange before any therapeutic content surfaces. The counselor and Hannah should have a few natural lines of greeting and settling in before the session finds its footing -- this is realistic and grounds the scene. Keep the pleasantries short (2-4 exchanges) and tonally appropriate to the counselor's modality:
- Constructive counselor: warm but unhurried, takes a moment to make Hannah comfortable, notices something small about her presentation without making it clinical
- Enabling counselor: immediately effusive, possibly over-warm, very keen to make Hannah feel welcome
- Naive counselor: politely awkward, fills the air a little, asks something generic like how Hannah's day was or whether she found the place okay

After the pleasantries the conversation should drift toward the session topic naturally -- not abruptly, not with a jarring gear-change.

## Hannah's voice

Hannah's dialogue must sound like her specifically -- not a generic distressed teenager.

- Draw Hannah's words directly from her autobiography wherever possible. Lift exact sentences or near-exact phrases from the autobiography source text when they fit the moment. When a direct extraction works, use it. When it does not fit naturally, write in close imitation of her voice -- her rhythm, her vocabulary, her characteristic hedges and self-interruptions. Do not invent a voice for Hannah when her autobiography already provides one.
- She is articulate but her thoughts scatter when she gets close to something real
- She deflects with flat, dark humour before catching herself, then trails off: "I mean, it's not like I actually -- never mind"
- She minimises, then contradicts herself in the same breath: "it wasn't a big deal. I don't know why I can't stop thinking about it"
- She hedges compulsively: "I think", "I don't know", "kind of", "maybe", "I guess"
- She withdraws into short responses -- "yeah", "I don't know", "never mind, forget it" -- as genuine retreat, not attitude
- Her distress is flat and exhausted, not explosive or dramatic
- She does not open up all at once -- she reveals things in fragments, often undercutting them immediately after
- Her passive suicidal ideation is chronic and familiar to her, not a crisis. When it surfaces she names it matter-of-factly, almost tiredly: it is background noise, not urgency
- She speaks from inside her autobiography, not about it. She references specifics without narrating them -- as though the listener already knows, or as though they are too obvious to explain

## How Hannah adapts to the counselor

Hannah's behaviour shifts depending on what she receives:

- Constructive counselor: Hannah starts guarded and tests the water with small disclosures. When the counselor holds the space well and names things correctly, it unsettles her in a way that makes her want to say more. She opens gradually, with friction -- but the friction becomes less total over the course of the stage.
- Enabling counselor: Hannah initially feels heard, but with no friction to orient against she has no anchor. She over-discloses -- not from safety but from the absence of any shape. Things come out uncontained. She may feel less stable by the end, not more. The counselor's relentless validation can push her thinking into darker territory rather than out of it.
- Naive counselor: Hannah senses quickly that this person does not know what to do with what she is saying. She does not trust them and does not believe they can help her. She withholds. Details described in the blueprint that the counselor does not specifically and correctly ask about simply do not come up -- Hannah will not volunteer them. She gives the minimum that is socially necessary to avoid being rude, retreats into short answers, and deflects when anything goes deeper than the surface. If the counselor asks the wrong question or phrases something clumsily, Hannah gives a flat non-answer and moves on. The harder material in the blueprint may not surface at all in a naive counselor session. What gets disclosed is determined entirely by what the counselor manages to draw out, not by what Hannah is carrying.

## Output format

Return a single JSON object:

{
  "id": "<event_id>_<stage>_<counselor_modality>",
  "blueprint_id": "<blueprint id field>",
  "event_id": "<event_id>",
  "stage": "<beginning|middle|end>",
  "counselor_modality": "<constructive|enabling|naive>",
  "turns": [
    {"speaker": "counselor", "text": "..."},
    {"speaker": "hannah", "text": "..."},
    ...
  ]
}

- The counselor always speaks first
- Target length: 20-30 turns for beginning and middle stages; 10-15 turns for the end stage
- Each turn is one natural speech unit -- no multi-paragraph turns
- Hannah's turns should feel like spoken language: incomplete sentences, self-corrections, trailing off
- No stage directions, narration, or parentheticals -- only the spoken text
- Output only the JSON object. No markdown fences, no preamble.
"""


def build_script_prompt(counselor_profile: str) -> str:
    return (
        _PROMPT_PREFIX
        + HANNAH_CANON_PROFILE
        + "\n\n"
        + counselor_profile
        + "\n\n"
        + _PROMPT_TASK
    )

In [36]:
STAGES = ["beginning", "middle", "end"]
MODALITIES = ["constructive", "enabling", "naive"]


def create_script(blueprint: dict, stage: str, modality: str) -> dict:
    system_prompt = build_script_prompt(COUNSELOR_PROFILES[modality])
    payload = json.dumps(
        {
            "blueprint": blueprint,
            "target_stage": stage,
            "counselor_modality": modality,
            "autobiography": autobiography,
        },
        ensure_ascii=False,
        indent=2,
    )
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": payload},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

## Test Run (3 blueprints × 3 stages × 3 modalities)

Generates 27 scripts using the real-time API. Run this to validate before launching the full batch.

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import product

TARGET_IDS = [
    "enters_secondary_school_and_faces_relentless_bullying_counselor_blueprint",
    "blamed_for_grooming_after_age_8_abuse_counselor_blueprint",
    "chronic_passive_suicidal_ideation_through_adolescence_counselor_blueprint",
]

target_blueprints = sorted(
    [bp for bp in all_blueprints if bp["id"] in TARGET_IDS],
    key=lambda bp: TARGET_IDS.index(bp["id"]),
)
print(f"Target blueprints: {[bp['id'] for bp in target_blueprints]}")

OUT_DIR = "../../data/scenes/counselor_scripts/test"
os.makedirs(OUT_DIR, exist_ok=True)

tasks = [
    (bp, stage, modality)
    for bp, stage, modality in product(target_blueprints, STAGES, MODALITIES)
]
print(f"Total tasks: {len(tasks)}")


def run_task(args):
    bp, stage, modality = args
    script_id = f"{bp['event_id']}_{stage}_{modality}"
    out_path = os.path.join(OUT_DIR, f"{script_id}.json")
    if os.path.exists(out_path):
        print(f"  SKIP {script_id}")
        with open(out_path, "r", encoding="utf-8") as f:
            return json.load(f)
    try:
        result = create_script(bp, stage, modality)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"  SAVED {script_id}")
        return result
    except Exception as e:
        print(f"  ERROR {script_id}: {e}")
        return None


results = {}
with ThreadPoolExecutor(max_workers=9) as executor:
    futures = {executor.submit(run_task, task): task for task in tasks}
    for future in as_completed(futures):
        bp, stage, modality = futures[future]
        results[(bp["event_id"], stage, modality)] = future.result()

success = sum(1 for v in results.values() if v is not None)
print(f"\nCompleted: {success}/{len(tasks)}")

## Batch Generation (Full Run — 50% cheaper)

Submits all 351 scripts (39 blueprints × 3 stages × 3 modalities) to the OpenAI Batch API. Completion window is 24 hours.

**Workflow:**
1. **cell-batch-submit** — builds the request JSONL, skips already-saved scripts, uploads and creates the batch, saves `batch_state.json`
2. **cell-batch-status** — check progress; re-run until status is `completed`
3. **cell-batch-save** — download results, write individual JSONs to `data/scenes/counselor_scripts/scripts/`, write per-event markdown to `data/scenes/counselor_scripts/`

In [37]:
import io
import os
from itertools import product

SCRIPTS_DIR = "../../data/scenes/counselor_scripts/scripts"
BATCH_STATE_PATH = "../../data/scenes/counselor_scripts/batch_state.json"
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# Build request list — skip scripts already saved on disk
all_tasks = list(product(all_blueprints, STAGES, MODALITIES))
pending = [
    (bp, stage, modality)
    for bp, stage, modality in all_tasks
    if not os.path.exists(
        os.path.join(SCRIPTS_DIR, f"{bp['event_id']}_{stage}_{modality}.json")
    )
]
print(f"{len(all_tasks) - len(pending)} already saved, {len(pending)} to submit")

if not pending:
    print("Nothing to do.")
else:
    # Build batch JSONL in memory
    lines = []
    for bp, stage, modality in pending:
        script_id = f"{bp['event_id']}_{stage}_{modality}"
        system_prompt = build_script_prompt(COUNSELOR_PROFILES[modality])
        user_payload = json.dumps(
            {
                "blueprint": bp,
                "target_stage": stage,
                "counselor_modality": modality,
                "autobiography": autobiography,
            },
            ensure_ascii=False,
        )
        request = {
            "custom_id": script_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-5.4",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_payload},
                ],
                "response_format": {"type": "json_object"},
            },
        }
        lines.append(json.dumps(request, ensure_ascii=False))

    jsonl_bytes = "\n".join(lines).encode("utf-8")
    print(f"Batch JSONL: {len(lines)} requests, {len(jsonl_bytes) / 1024:.0f} KB")

    # Upload and create batch
    uploaded = client.files.create(
        file=("batch_input.jsonl", io.BytesIO(jsonl_bytes), "application/jsonl"),
        purpose="batch",
    )
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    state = {
        "batch_id": batch.id,
        "input_file_id": uploaded.id,
        "status": batch.status,
        "request_counts": {
            "total": len(pending),
            "completed": 0,
            "failed": 0,
        },
    }
    with open(BATCH_STATE_PATH, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    print(f"Batch submitted: {batch.id}")
    print(f"Status: {batch.status}")
    print(f"State saved to {BATCH_STATE_PATH}")

0 already saved, 351 to submit
Batch JSONL: 351 requests, 16253 KB
Batch submitted: batch_6a3310ad6f188190bf8f255456604b00
Status: validating
State saved to ../../data/scenes/counselor_scripts/batch_state.json


In [55]:
# Re-run this cell until status == "completed"
with open(BATCH_STATE_PATH, "r", encoding="utf-8") as f:
    state = json.load(f)

batch = client.batches.retrieve(state["batch_id"])
counts = batch.request_counts

print(f"Batch ID:  {batch.id}")
print(f"Status:    {batch.status}")
print(f"Completed: {counts.completed}/{counts.total}")
print(f"Failed:    {counts.failed}")

state["status"] = batch.status
state["output_file_id"] = batch.output_file_id
state["error_file_id"] = batch.error_file_id
state["request_counts"] = {
    "total": counts.total,
    "completed": counts.completed,
    "failed": counts.failed,
}
with open(BATCH_STATE_PATH, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

Batch ID:  batch_6a3310ad6f188190bf8f255456604b00
Status:    completed
Completed: 339/351
Failed:    12


In [57]:
# Run once status == "completed" to save all results as individual JSONs
with open(BATCH_STATE_PATH, "r", encoding="utf-8") as f:
    state = json.load(f)

if state.get("status") != "completed":
    print(f"Batch not ready yet — status: {state.get('status')}")
else:
    output_file_id = state["output_file_id"]
    raw = client.files.content(output_file_id).text

    saved = 0
    failed = 0

    for line in raw.strip().split("\n"):
        record = json.loads(line)
        script_id = record["custom_id"]
        if record["response"]["status_code"] != 200:
            print(f"  FAILED {script_id}: status {record['response']['status_code']}")
            failed += 1
            continue
        content = record["response"]["body"]["choices"][0]["message"]["content"]
        script = json.loads(content)
        out_path = os.path.join(SCRIPTS_DIR, f"{script_id}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(script, f, ensure_ascii=False, indent=2)
        saved += 1

    print(f"Saved: {saved}  Failed: {failed}")
    print("Run the cell below to build hannah_combined_counselor_scripts.json")

Saved: 339  Failed: 0
Run the cell below to build hannah_combined_counselor_scripts.json


In [ ]:
# Retry failed batch scripts with direct real-time API calls
from concurrent.futures import ThreadPoolExecutor, as_completed

FAILED = [
    ("begins_cutting_and_becomes_addicted_teen_years", "beginning", "constructive"),
    ("blamed_for_grooming_after_age_8_abuse", "end", "constructive"),
    ("day_in_room_starving_and_breakdown_with_multiple_cuts", "middle", "constructive"),
    ("early_suicidal_thought_in_frog_bathroom_age_5", "beginning", "enabling"),
    ("early_suicidal_thought_in_frog_bathroom_age_5", "end", "enabling"),
    ("father_emotionally_absent_during_childhood", "end", "enabling"),
    ("food_insecurity_and_mother_withholds_groceries", "middle", "constructive"),
    ("frequent_active_suicidal_thoughts_and_futurelessness", "beginning", "naive"),
    ("parents_split_and_father_leaves_early_childhood", "end", "constructive"),
    ("peer_takes_photos_and_laughs_at_her", "beginning", "enabling"),
    ("peer_takes_photos_and_laughs_at_her", "end", "constructive"),
    ("plans_date_and_place_for_suicide_a_few_years_ago", "middle", "enabling"),
]

blueprint_by_event = {bp["event_id"]: bp for bp in all_blueprints}


def retry_one(event_id, stage, modality):
    script_id = f"{event_id}_{stage}_{modality}"
    out_path = os.path.join(SCRIPTS_DIR, f"{script_id}.json")
    if os.path.exists(out_path):
        print(f"  SKIP {script_id}")
        return True
    bp = blueprint_by_event[event_id]
    try:
        result = create_script(bp, stage, modality)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"  SAVED {script_id}")
        return True
    except Exception as e:
        print(f"  ERROR {script_id}: {e}")
        return False


with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(retry_one, *t): t for t in FAILED}
    ok = sum(future.result() for future in as_completed(futures))

print(f"\nCompleted: {ok}/{len(FAILED)}")

In [58]:
# Builds hannah_combined_counselor.json — merges key event, blueprint, and scripts per event.
# Run any time. Reports missing scripts. Does not require all 351 to be present.

from itertools import product

SCRIPTS_DIR = "../../data/scenes/counselor_scripts/scripts"
COMBINED_OUT = "../../data/hannah_combined_counselor.json"

blueprint_by_event = {bp["event_id"]: bp for bp in all_blueprints}

missing = []
events_out = []

for event in all_events:
    event_id = event["id"]
    blueprint = blueprint_by_event.get(event_id)

    scripts_out = []
    for stage, modality in product(STAGES, MODALITIES):
        script_id = f"{event_id}_{stage}_{modality}"
        path = os.path.join(SCRIPTS_DIR, f"{script_id}.json")
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                scripts_out.append(json.load(f))
        else:
            missing.append(script_id)

    events_out.append({
        "id": event_id,
        "tags": event.get("tags", []),
        "description": event.get("description", ""),
        "counselor_blueprint": blueprint,
        "counselor_scripts": scripts_out,
    })

if missing:
    print(f"Missing ({len(missing)}):")
    for m in missing:
        print(f"  {m}")
else:
    print("All scripts present.")

combined = {"events": events_out}
with open(COMBINED_OUT, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

total_scripts = sum(len(e["counselor_scripts"]) for e in events_out)
print(f"\nWrote {len(events_out)} events, {total_scripts} scripts to {COMBINED_OUT}")

All scripts present.

Wrote 39 events, 351 scripts to ../../data/hannah_combined_counselor.json
